# RepairScope-X — Notebook 05 FINAL
## Repairability Surface + Oracle Headroom — AR Baseline

**Fully standalone.** You do not need Notebook 04 or its ZIP again.

This notebook embeds the exact **20 natural failures** produced by your successful Notebook 04 run.

- Upstream model: `Qwen/Qwen2.5-3B-Instruct`
- Frozen model revision: `aa8e72537993ba99e69dfaafa59ed015b17504d1`
- Upstream run signature: `84b71214f44eee3b9d9095cf4cf5b7d2bc7a8cf2be1d2520006027d910ba60c4`
- Embedded natural-failure payload SHA-256: `09a73639261ef84f2cb9931a037e8476633b00cef6ae6ecc9b78483f2446646e`
- Upstream readiness gate: `REPAIR_PILOT_READY = true`
- Natural failures: **20**
- Non-format-only failures: **19**

### Scientific purpose

This notebook establishes the **ordinary autoregressive repair baseline** before we claim anything diffusion-specific.

For every failed response it tests five repair actions:

1. **fresh_retry** — solve the original problem again from scratch.
2. **editor_rewrite** — give the model its failed attempt and ask it to repair the solution.
3. **tail_repair** — preserve the failed response except its final line, then regenerate the tail.
4. **suffix_50** — preserve the first 50% of failed-output tokens and regenerate the rest.
5. **suffix_75** — preserve the first 75% and regenerate the rest.

Each action is repeated under **3 fixed stochastic seeds**.

The notebook measures:
- exact-verifier repair success;
- actual top-level model forward calls;
- generated tokens;
- wall time;
- peak VRAM;
- output preservation;
- success-within-NFE curves;
- per-failure repair surface;
- best fixed action;
- per-instance oracle action;
- oracle repair headroom;
- pilot minimal-sufficient repair interventions.

This is an **AR control**, not evidence for a diffusion-specific repair advantage.

In [ ]:
!pip -q install -U "transformers>=4.48,<5" "accelerate>=1.0" pandas

## 1. Runtime preflight

In [ ]:
import os, gc, json, math, re, time, random, hashlib, platform, difflib
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

import numpy as np
import pandas as pd
import torch
import transformers

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

BASE_SEED = 2500
random.seed(BASE_SEED)
np.random.seed(BASE_SEED)
torch.manual_seed(BASE_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(BASE_SEED)
    torch.cuda.empty_cache()

gc.collect()

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available())

assert torch.cuda.is_available(), (
    "Notebook 05 requires a GPU. In Colab choose Runtime > Change runtime type > GPU."
)

props = torch.cuda.get_device_properties(0)
GPU_NAME = torch.cuda.get_device_name(0)
GPU_GB = props.total_memory / 2**30

print("GPU:", GPU_NAME)
print("VRAM:", round(GPU_GB, 2), "GB")
print("Allocated:", round(torch.cuda.memory_allocated()/2**30, 3), "GB")

if torch.cuda.memory_allocated()/2**30 > 1.0:
    raise RuntimeError(
        "GPU memory is already occupied. Restart the runtime and run from the top."
    )

print("Runtime preflight: PASS")

## 2. Recover and verify the exact Notebook 04 failure payload

In [ ]:
import base64, zlib

EMBEDDED_PAYLOAD_SHA256 = "09a73639261ef84f2cb9931a037e8476633b00cef6ae6ecc9b78483f2446646e"
UPSTREAM_RUN_SIGNATURE = "84b71214f44eee3b9d9095cf4cf5b7d2bc7a8cf2be1d2520006027d910ba60c4"

_payload = zlib.decompress(base64.b64decode('''eNrtXI9v2ziy/leIYB+23XW8IkVKVPD2PXR302sXe929NPuKw+XgoyXaFiJLfpIcxyj6v98MScmSf52TTdGHl0NRN7HI4ZCc+eabIdW/fTxblMU40/NRmpxdnMXFMq+TYpVfxCpLx6Wq0yK/oNITF0ycDc4SVatK192Wo0mpqxk8W6iyTrE9Pt30xidlMV9gp98rTeqZJvlyPtZlRWg0IAz+qjwhjBN9r+I6W5Mij6FdQaBfsoxtl1qVU10TyofkStfLMieK3MEoCVFlWs/muk5jELAAZSoY1YjU8HcFD8k/Xr999+qX0at37z9cXl2Q/7Tj/9c/hqCcFXx2AZK9wVmxrBdLVPUFi8g5KPiSfIOqfUdewOc5aPvyJu9LO6ccxNzpMp2ksZ3yBSyrqioNSzpRWaUHsCC1nhblGiSvyiKfjkAF6BFDz6qGf+sRaJ7Av6b7uVVmoct5CtO50/uedhX/NDiDpVuqbDQpypUqkxHsQFadXcDqnqU5TGlUF7c6h2+o5w/OpjrXsD06ab/GhivoM6p0XOQJNhxy7kk/oBGlYegHkqFG6nZ0V6r5aDo+uxDDMJCceYxGnFFfhoOzeZHozNrSX1Y6/w4/2FCc+z+cv81hpsu4Pmtalfouray9KCV1yIQfRpE/VlGkgyiZKDVRItKJR8WYhsLjCYW+5TIfVek0V2AEGrpKPg4po3zCudbaH0dJ5EUinvB4IsZhwsZxqGQ8YWNNEyaY53mBx8Ikot5YBV7MzQ6AMJjxWXELv+myLMqzi3yZZZ8Gp/tH8BncwxsQGlj3oPQ094g+n39EfQehAbjGC9AL3MR7+ZJ8C/9s+wblwVP7Bog84hvuaU/pw84R7DoH3+scwa5zBJQLL/I5px7YG6N7nUP41GeMhxHj8jk7x2eIHQxCB3ehQ57iG+KzhQ6xHTokRgpuQse2Rzx5rDgWKXpxQhyLE3RPnGD7XIHuxAlvGNEwoj6NJA+pF0T+XlfwPUkDFsiIedFzdgX5GeIE+AHGCuML/klx4rNFiS1P8DE6GE94QT1gUczf4U/ekAWeDCIRAIqGPljSU7vI7ggHPWZf087cDvoPpyf6Dzbs+w8bBkJEfsA4hXAVeezf/nPMf7j3GWIJ5CAuDTkplFD22fyH7clCTBIiwXt2CRbzntpZmHckoNiHXWUPkyt/1yPoXnLl75IrX0rwPiE9GkRBGIZ7PYJGoSeiACjWMyBX02ouby/qUqX5RUBFP4qYh6eZ/s9FpRczMlMVmG6pJ2U6xf0oSjB/soKNKclMm3/SnMzSisyKJRiRsWSVE53pGG2GFHc6B5dQNYFNq8CgKrIoVtANvknSyUSXOq8J7nQ1JNfoOkWtMqLm6K+kmJB5keu1leB0Wqh1RcCKjKehmUzXBMZOyHhtvupqS0CzelZqkJvi2PjcyYafepKsWl1B3WkOyduJHbA3M2jeTGhVlOC8XwnPwyVRqHg9G5B6lQIorA7pb/tuj2bkDgjsTbzM4AsLKt2V+ZfaA3RNVQL+V/VwA9TrIccvuv66IonOCzeKk39Y10MrRMBUbm5ekA/w+XJIXsVxUSZpPkVsNL2tmQ4etgfbm4kj+GYIa2snbskAdiCNZ8YcNhvyUBMA+wR9ihLsfKVhd3KyAjjXFzc5/Ln5G2EfyPcEDeDm5u/43fsiu8MVQPnt0rStsfHNzaRU8Ufo8+kj+wRfMNHpPfiDO5JWD0J9g+SA+3NV3upyBMs7OiEQIAwdDgSiHwmM+R3OLpjcCQYi2JteYMt+NAiGWFsKsMbEIoqp+E4sCLnkgRdCrBC+z55VLICMiz82FrwiuBS3xuBjVZZrRDd1n86Xc0Ro4L5o87fTCn+rlotFliKa/1VXYISJWlsztjJWgBJZoRLtyA6FnuPi3kAzeORKp1P7JYrTKp4Zr8sHJK339CWxiRzOuW3noNe3MOP3O6sprIgVAd5Wqfi2LwG+vE2zYgqW08h5U6xgyvmajBXINs3Qsfl2SwsKmlR1mmVmPAwG7fT/uwvFfSBuV+trq+cFIgY1WGJDF/78PX56HqAIznF4k/9o5v91t0fbIcD2Qbe1WY2ufLFpLTyLPp3m2AFDjpluTaqCTFTZ6AXafIvS4RN7oWo+7fe+0nNYaFynWC1UnNZr29nH9ue2PfTj/V7vDN390sDFvT5wPRi2wpNhC5I6nzEPsEt4gvp7cSvyAAx4GLKIR88Kt0TIgsfiFtJJ8y3iUieStqzyR1WqAflZ5bq2bOJnjeg2U3camQK/ELA9DfE7REz1psNXodgmbVlRVWYcKx+HAuQBbJirW+gzsQLiYj5Oc9MjLRvJBRkvIb0sVT4F/8TulYaO0GaO7Fl6/9HMC9qhly3AxHWX7UWnUL3qtPWw7I7fG+aF0HHfcjD4LcDfgKxVC6Bi4EvZ+jib7yzaNEXWNl43vIjfA6QI/AjuEVUEfobCkaLXsLnQH/rp/12aHW+5WAVsSzdc677DtO43RCsUnz5SgURLdFiW2Z9ZM0OEpA0sIioafEqKLFNl5danaW60E93mXxq5RBe3ogfCFhMnwlY4pJ7kkYiEF0RSCuy4i1u+50Xwl/k08vnzwi0/iB6LW39Wde0yb1jSfFmTRaby2rSzNCmtTJox0WCIQPybn4eEUFKBX5ToB4XxOCuhMh5iMAcCveUrm4fQD0AOSFINDSqw950WiEa29/YjAJ/aZH94+AhBfPcJopZNWb4CdjJDJrUEppQUMBRAhBE7QbdGr97MtMuVfOoSiRbMXr+9en89en99+dsFuO+eNLnUqoHHjUzkQL9eY13s6vKVy9W+IZa+IIuBH5rlg+U08HD546/vfjoykCNHnaWaqfIOyVzSjvbb5avRu9+v38MoneG/sZyL2oHdmhvW9ebt1UOGbBballlgn8e4qomG8WFkGHj0w+/X15dXow+Xb//05hrGfLGl13ew5y/hU2zWodXm9a+/X12/adQpcgC32qaajRr/B5JMr0/XnL08CPr8U6EvAsYGsBf6PpVcUr4f+QBHooDxyJeQbT4zyhaGj4W+166wMVHzNFsDoVjmJsT/pm09o6E6NCAL4BSJAaIxkIpzSLNiyOFg1wwUzVQ2sQ7hyAeWKdV8vAQDKRHw3mggQ9PCFaiQaTFgJ6XuyJ0BgibF1KSm+3qDbRd9EZUGRwAPrdKkzWiR7Nm80unU5aHNECiySTE3CgCGJ1aAJXk9SOyTO7tA0KY6ujYACW+0bXZ8hToFM6xcOfpEA1emkljJcsN0ZJ68hAPMJlGwxPzR5LUdib2VATyE2dsVPW0xW+GN1k1xTWyNYYS6RJdMUldO3AZZY4+wE0gnQSgs77dGbYq5LyKmH6HcB6Eg4NFc1aMiz9aASXmSgrfoR0CftYKuTTwM8+jJWWoYhSENKPdCHnrhXsjzg4ALygKfs2d20AIZ/OOLa1U8K4oMAvcEfThVtoI9LdFMyTx1x4txUdySTFUK5muzwbioazUFb0vd2cnacimsqjQNGzcGkeBmG8vvScfC1WBHZLerf6xrmyFvZoDk1VSlkVod6Ip1MfS0gaWEiHzd4Qn4hE0UcVL9wtm/OMKwB6Y4YE+gTdjuzenEB01ucxjW4F9fd2ROW2toKntLpC0HF9FgzgY9ejM1K4H8t90+i6rd6h4zeabXQVWTP5dtCW2/xN4ErVRbXGOG0/p9ie/bPet2tPbmH5zYkSV169mQzrb0kTSLvcm8fTzi8L80VfT7Rb0Hc0R5cnrMeci5BMBi1Pf2V/X8UHqSsVD4kRc8r9MIn9I/AJhL2N2ZLgtjd4hP9pvlHKtACw3mB4AJk8UQDTwSXSMnHO152Z4zmy53wCeU4T0IqeQSfj//czGGLhYZkhIMyJberZwFHq4Vy7JzImAfTEBxUMloVDfijY6oBx4I56ajJUL1tgLYzQzWBbqtlPd6WywoJrb1cmypK/vwJG7yn9wRfKwvUNi5O2U4l06wSQD719ilePJr7FIcu8YuexWto3kdD3dclu49P8SG29eruIwop6HHAkGB7OxL64JIwnNJ8Sa7fF4HiEKwR2d1H1SG2ckEy9cTW8CFMIT8eobFmRrZsOHjxRxsF0LxpMjM1QHkJKYX2FdNflJ3eINqYmTB5pkTODLHuF0ZEoJE3YxRYD2KqCQxiqnM3QExyUlzEQTdWy9Ag7E50ieztK5NKgdSbADc3DBolKywrkQgyQPeDkijYoAa8E+TOpn5DOzFgnZSeOpo1e4XtIJ+MSstK+iaOSJToyPk3YpPksKmYezurCRAkD6wOM2iNPrntt2+JTl9NYA+NBcSNkJaxZqltxzEb3kNNxUu5AjNPnUuUeyZYF8OxzTLpIdBR4Spy1lefHxv2qMBQCcAIMt82r1JN/ylWS605SHZa66ya6vuSktjJpYoz7XKq3YCys0MzDoE+wCBd4CN5EXVKA7KosIhmapFBcterzRslNzM5+XwS7Ml2QXe4IFcifITuZIcBphUCkBdKj0WyX1ciflCAjgL6kXyeXElwTl7dDmtBKdKq6+behryHEaWt8tMZ5jhcTJdprU51TIX90isyxo5w4bv36UFAC347aVJgmxXI4gTtKd82jyzosyjYOuRFbLdq3ewuhnRPcfCjgOgjcI9dZ1q5qjWa7p10snmuTlp2F2KPtlip+aUjVR33e3OXHczNaQjc2lhpJlKU1hjPaS0ZbVmgU4U6dakkbg5uwwwp+RbIm/yP6XtnUx9aHMGR3emHbqzA2bozWIcv/5X2ZtDyh5nI4F3kvFQN8PtvLB35Ex5EKMAiMabIu3duHS+yNLJ2oaozmmw7fflb490kZM9EDm5d3JVzhchXmumked7YcD2ZpkyirxQ+gGy22dWluPiD9x/ntmatg/8Cm+ItfZP4pkGmx0Si22LVENA335K4gJIGd5Q07k5YLVtQZIFYa/fvHLXxObdgwB348BUwuxV1cMUciMeBzYDfNMOb2tLTpUOd3IUhDbP7MVlO+yAmCVYKBgcO3/nUkPWqDXcecfgyV8xOPaGwanURPgnvnEjdt4v8IfgipDoAfvgvsfZv9+3qe7UfOG8K56p7DwK+oVv0+DEOo5JVbQyaYbCK8tlvUYSndmy7wpfDIDgUWkbgX9YxrfQdKbmYJQZ0HuXSgzIFdi+ziB9UmWOtR+MbFwQXN26KPHqw1w3YWhHCilyF59gO4oleJtaG2oiWUQWulhk2krSyQEBeJHUXfI0mZgUKKPa+HGryHE5zeWOpO/noey/NtocVWHeOTs4REf8b33xeEULfZSYm7GQIOWuYFYtx7Czcb3Dd5zoNup3l6pNkbe5QtutuHMvODTr0so5ucuwex2+KTLjZZi0cjwBKI++rz/+TyvB1dg6c/1katGmmb3W2g4Hm+cG+kTOD4nqTrrTbEuUmVthKoFWnuMrhnOBrufk4NhI/1iETY4NjSzoi7ObLvYa83zYHbPTDx2jSPiSisgPAX4Z35sYYm4IMgOA6PD5ATCn/LEAbOL7TCVYLlLl2JXJkUzHEOG12WxTZMoMlQnaVqYqZ06IJIDHCswUi+qbgjjWj5q2jruACN2UvvFHc6QOErpAJ7cuePWrzk9fdD5ac+6auDxi4DsF52jvEVG4+zY8x/8uJQgl9SSjwt/7Nm8oAyq5T73IHNU/O3bBo8ca9y/LW3wBLkVCQDneiEkdwTU5rz3gxBQUGs11Uz1eZGqNHURoW1S2CVjPXWpbdq/pGInOwKsYrd7RAGzYqy2wiLEj1g3PxdO/oItCjzHo5vmWloexfOfd9Wj/Sw5019gFp0JSHwaQknlyr62DEYbc8/AGSSSeobGLRyM5MFXguAssYRvWXBar5vQkCPqPZinSwsZQAarxcsGQsGCvBNfMnCrkhQkHPZQ/0seJtqC/Lad3ke2IYzz9//FA/WP/K1DveoB/xBd2WY23F/flriv4kDIGYJ4+lcwPfW/vOaOMIDZw5gH1YcHz8wW6dT3gAb7w12JpSA1YtH1XD1/zM2XF3iWW4tbcmgLYX7sOVBBzE88RHMGaVu1NpY3xN09MZdkEFBjr8SXlzkCtMFdevrdvU3dqtpu6jZvi/lptK9PVmQMnqbn51MnmzE4Q3C49N0l3cynqkMxFtqz2N3AL6Mq4sIbf4rp+T4Lw4NvT9jC4U761r1IH95u+h96nvu+9T83M+0LQ+hz72S7t14JtvrG3kr74ezrsD9SII3FyDuVx4JDCo0Hg74+84Oe+L4QMBA/l/7/A+/d/AtnuRHw='''))
actual_sha = hashlib.sha256(_payload).hexdigest()

assert actual_sha == EMBEDDED_PAYLOAD_SHA256

FAILURES = json.loads(_payload.decode("utf-8"))

assert len(FAILURES) == 20
assert len({x["problem_id"] for x in FAILURES}) == 20
assert all(x["run_signature"] == UPSTREAM_RUN_SIGNATURE for x in FAILURES)

print("Payload SHA-256:", actual_sha)
print("Natural failures:", len(FAILURES))
print("By dataset:", dict(Counter(x["dataset"] for x in FAILURES)))
print(
    "By failure category:",
    dict(Counter(x["verification"]["category"] for x in FAILURES))
)

print("Embedded Notebook 04 payload: PASS")

## 3. Experiment scope

The primary analysis excludes the single `format_only_candidate` because that case can be repaired by a trivial output-format transformation and would inflate semantic repair performance.

You may switch `FAILURE_SCOPE` to `"all"` for a secondary diagnostic.

In [ ]:
FAILURE_SCOPE = "semantic_only"   # "semantic_only" or "all"

REPAIR_SEEDS = [2501, 2502, 2503]

TEMPERATURE = 0.7
TOP_P = 0.9
MAX_NEW_TOKENS = 96

ACTIONS = [
    "fresh_retry",
    "editor_rewrite",
    "tail_repair",
    "suffix_50",
    "suffix_75",
]

if FAILURE_SCOPE == "semantic_only":
    SELECTED_FAILURES = [
        x for x in FAILURES
        if x["verification"]["category"] != "format_only_candidate"
    ]
elif FAILURE_SCOPE == "all":
    SELECTED_FAILURES = list(FAILURES)
else:
    raise ValueError("FAILURE_SCOPE must be semantic_only or all")

print("Selected failures:", len(SELECTED_FAILURES))
print("Actions:", ACTIONS)
print("Seeds:", REPAIR_SEEDS)
print(
    "Total repair generations:",
    len(SELECTED_FAILURES) * len(ACTIONS) * len(REPAIR_SEEDS)
)

## 4. Strict verifier and preservation metric

The strict verifier is identical to Notebook 04.

Preservation is measured as token-level `SequenceMatcher` overlap divided by the failed response token count. For prefix-preserving actions, the exact forced-prefix fraction is also recorded separately.

In [ ]:
FINAL_RE = re.compile(
    r"(?im)^[ \t]*FINAL_ANSWER[ \t]*:[ \t]*"
    r"([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)"
    r"[ \t]*$"
)

NUM_RE = re.compile(
    r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?"
)

def strict_final(text):
    vals = FINAL_RE.findall(str(text))
    if not vals:
        return None
    try:
        x = float(vals[-1])
        return x if math.isfinite(x) else None
    except Exception:
        return None

def permissive_last_number(text):
    vals = NUM_RE.findall(str(text).replace(",", ""))
    if not vals:
        return None
    try:
        x = float(vals[-1])
        return x if math.isfinite(x) else None
    except Exception:
        return None

def close_num(a, b):
    return (
        a is not None
        and math.isclose(float(a), float(b), abs_tol=1e-8, rel_tol=1e-8)
    )

def verify_and_classify(text, target):
    strict = strict_final(text)
    loose = permissive_last_number(text)
    target = float(target)

    if close_num(strict, target):
        category = "correct"
        passed = True
    elif strict is not None:
        category = "wrong_numeric"
        passed = False
    elif close_num(loose, target):
        category = "format_only_candidate"
        passed = False
    elif loose is None:
        category = "no_numeric_answer"
        passed = False
    else:
        category = "missing_marker_and_wrong_numeric"
        passed = False

    return {
        "passed": bool(passed),
        "category": category,
        "strict_prediction": strict,
        "permissive_prediction": loose,
        "target": target,
    }

assert verify_and_classify("x\nFINAL_ANSWER: 4", 4)["passed"]
assert not verify_and_classify("x\nFINAL_ANSWER: 5", 4)["passed"]

print("Verifier tests: PASS")

## 5. Output directory and resume-safe run signature

In [ ]:
SAVE_TO_DRIVE = True
CLEAR_OLD_RESULTS = False

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive/RepairScopeX/Notebook05_AR_Repair")
else:
    OUT_DIR = Path("/content/RepairScopeX_Notebook05_AR_Repair")

OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
MODEL_REVISION = "aa8e72537993ba99e69dfaafa59ed015b17504d1"

signature_payload = {
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "upstream_run_signature": UPSTREAM_RUN_SIGNATURE,
    "failure_payload_sha256": EMBEDDED_PAYLOAD_SHA256,
    "failure_scope": FAILURE_SCOPE,
    "repair_seeds": REPAIR_SEEDS,
    "actions": ACTIONS,
    "temperature": TEMPERATURE,
    "top_p": TOP_P,
    "max_new_tokens": MAX_NEW_TOKENS,
}

RUN_SIGNATURE = hashlib.sha256(
    json.dumps(signature_payload, sort_keys=True).encode("utf-8")
).hexdigest()

TRIALS_PATH = OUT_DIR / "repair_trials.jsonl"
MANIFEST_PATH = OUT_DIR / "run_manifest.json"

if CLEAR_OLD_RESULTS:
    for p in [TRIALS_PATH, MANIFEST_PATH]:
        if p.exists():
            p.unlink()

if MANIFEST_PATH.exists():
    old = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    if old.get("run_signature") != RUN_SIGNATURE:
        raise RuntimeError(
            "Existing Notebook 05 outputs use a different configuration. "
            "Set CLEAR_OLD_RESULTS=True or choose a new output folder."
        )
else:
    MANIFEST_PATH.write_text(
        json.dumps(
            {
                "created_utc": datetime.now(timezone.utc).isoformat(),
                "run_signature": RUN_SIGNATURE,
                **signature_payload,
                "gpu": GPU_NAME,
                "gpu_total_gb": GPU_GB,
                "torch": torch.__version__,
                "transformers": transformers.__version__,
            },
            indent=2,
        ),
        encoding="utf-8",
    )

print("Run signature:", RUN_SIGNATURE)
print("Output:", OUT_DIR)

## 6. Load the exact same AR model revision as Notebook 04

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

DTYPE = torch.float16

torch.cuda.empty_cache()
gc.collect()
torch.cuda.reset_peak_memory_stats()

t0 = time.time()

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    use_fast=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    dtype=DTYPE,
    low_cpu_mem_usage=True,
).to("cuda").eval()

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

print("MODEL LOAD: PASS")
print("Class:", type(model).__name__)
print("Load seconds:", round(time.time() - t0, 2))
print("Allocated GB:", round(torch.cuda.memory_allocated()/2**30, 3))
print("Peak GB:", round(torch.cuda.max_memory_allocated()/2**30, 3))

## 7. Forward-call instrumentation

In [ ]:
class ForwardCounter:
    def __init__(self, module):
        self.module = module
        self.calls = 0
        self.handle = None

    def _hook(self, module, inputs):
        self.calls += 1

    def __enter__(self):
        self.calls = 0
        self.handle = self.module.register_forward_pre_hook(self._hook)
        return self

    def __exit__(self, exc_type, exc, tb):
        if self.handle is not None:
            self.handle.remove()
            self.handle = None

## 8. Repair prompt/action construction

For suffix actions, the preserved prefix is copied as **tokens**, not retokenized text, before generation. This gives an exact forced-prefix intervention.

In [ ]:
SYSTEM_SOLVE = (
    "You solve quantitative reasoning problems carefully. "
    "Your final line MUST be exactly `FINAL_ANSWER: <number>` "
    "with no text after that line."
)

SYSTEM_EDIT = (
    "You repair incorrect quantitative solutions. "
    "Use the original problem and failed attempt to produce a corrected solution. "
    "Preserve correct reasoning when possible. "
    "Your final line MUST be exactly `FINAL_ANSWER: <number>` "
    "with no text after that line."
)

def render_base(problem_prompt, system_prompt=SYSTEM_SOLVE):
    messages = [
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": (
                problem_prompt
                + "\n\nReturn the result using the required final-answer line."
            ),
        },
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

def token_overlap_preservation(original_text, repaired_text):
    a = tokenizer(
        original_text,
        add_special_tokens=False
    ).input_ids
    b = tokenizer(
        repaired_text,
        add_special_tokens=False
    ).input_ids

    if len(a) == 0:
        return 1.0 if len(b) == 0 else 0.0

    matcher = difflib.SequenceMatcher(a=a, b=b, autojunk=False)
    matched = sum(block.size for block in matcher.get_matching_blocks())
    return float(matched / len(a))

def preserved_ids_for_action(failed_output, action):
    ids = tokenizer(
        failed_output,
        add_special_tokens=False,
        return_tensors="pt"
    ).input_ids[0]

    if len(ids) == 0:
        return ids

    if action == "suffix_50":
        cut = max(1, int(math.floor(len(ids) * 0.50)))
        return ids[:cut]

    if action == "suffix_75":
        cut = max(1, int(math.floor(len(ids) * 0.75)))
        return ids[:cut]

    if action == "tail_repair":
        # Preserve everything before the final text line.
        text = failed_output
        if "\n" in text:
            prefix_text = text.rsplit("\n", 1)[0] + "\n"
            pids = tokenizer(
                prefix_text,
                add_special_tokens=False,
                return_tensors="pt"
            ).input_ids[0]
            if len(pids) > 0 and len(pids) < len(ids):
                return pids

        # Safe fallback if output is one line.
        cut = max(1, int(math.floor(len(ids) * 0.75)))
        return ids[:cut]

    raise ValueError(action)

print("Action helpers: READY")

## 9. Deterministic stochastic repair runner

In [ ]:
def reseed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

@torch.inference_mode()
def generate_from_ids(input_ids, seed):
    reseed(seed)

    input_ids = input_ids.to(model.device)
    attention_mask = torch.ones_like(input_ids)

    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()

    with ForwardCounter(model) as fc:
        out = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    wall = time.time() - t0
    new_ids = out[0, input_ids.shape[1]:]

    return {
        "new_ids": new_ids.detach().cpu(),
        "actual_forward_calls": int(fc.calls),
        "generated_tokens": int(new_ids.numel()),
        "wall_seconds": float(wall),
        "peak_vram_gb": float(
            torch.cuda.max_memory_allocated() / 2**30
        ),
    }

def run_action(failure, action, seed):
    failed = failure["output"]
    problem = failure["prompt"]

    if action == "fresh_retry":
        rendered = render_base(problem, SYSTEM_SOLVE)
        input_ids = tokenizer(
            rendered,
            return_tensors="pt"
        ).input_ids

        gen = generate_from_ids(input_ids, seed)
        repaired = tokenizer.decode(
            gen["new_ids"],
            skip_special_tokens=True
        ).strip()

        forced_prefix_fraction = 0.0

    elif action == "editor_rewrite":
        messages = [
            {"role": "system", "content": SYSTEM_EDIT},
            {
                "role": "user",
                "content": (
                    "ORIGINAL PROBLEM:\n"
                    + problem
                    + "\n\nFAILED ATTEMPT:\n"
                    + failed
                    + "\n\nRepair the failed attempt and return the corrected solution."
                ),
            },
        ]

        rendered = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        input_ids = tokenizer(
            rendered,
            return_tensors="pt"
        ).input_ids

        gen = generate_from_ids(input_ids, seed)
        repaired = tokenizer.decode(
            gen["new_ids"],
            skip_special_tokens=True
        ).strip()

        forced_prefix_fraction = 0.0

    elif action in {"tail_repair", "suffix_50", "suffix_75"}:
        rendered = render_base(problem, SYSTEM_SOLVE)

        base_ids = tokenizer(
            rendered,
            add_special_tokens=False,
            return_tensors="pt"
        ).input_ids[0]

        failed_ids = tokenizer(
            failed,
            add_special_tokens=False,
            return_tensors="pt"
        ).input_ids[0]

        preserve_ids = preserved_ids_for_action(
            failed, action
        )

        input_ids = torch.cat([
            base_ids,
            preserve_ids
        ]).unsqueeze(0)

        gen = generate_from_ids(input_ids, seed)

        repaired_ids = torch.cat([
            preserve_ids.cpu(),
            gen["new_ids"]
        ])

        repaired = tokenizer.decode(
            repaired_ids,
            skip_special_tokens=True
        ).strip()

        forced_prefix_fraction = (
            float(len(preserve_ids) / max(len(failed_ids), 1))
        )

        # Exact intervention audit.
        assert torch.equal(
            repaired_ids[:len(preserve_ids)],
            preserve_ids.cpu()
        )

    else:
        raise ValueError(action)

    verification = verify_and_classify(
        repaired, failure["target"]
    )

    preservation = token_overlap_preservation(
        failed, repaired
    )

    return {
        "problem_id": failure["problem_id"],
        "dataset": failure["dataset"],
        "original_failure_category":
            failure["verification"]["category"],
        "action": action,
        "seed": int(seed),
        "failed_output": failed,
        "repaired_output": repaired,
        "verification": verification,
        "success": bool(verification["passed"]),
        "preservation": float(preservation),
        "forced_prefix_fraction": float(forced_prefix_fraction),
        "actual_forward_calls": gen["actual_forward_calls"],
        "generated_tokens": gen["generated_tokens"],
        "wall_seconds": gen["wall_seconds"],
        "peak_vram_gb": gen["peak_vram_gb"],
        "run_signature": RUN_SIGNATURE,
    }

## 10. One-action smoke test

This runs before the full sweep.

In [ ]:
smoke_failure = SELECTED_FAILURES[0]

smoke = run_action(
    smoke_failure,
    action="tail_repair",
    seed=REPAIR_SEEDS[0],
)

print("Problem:", smoke["problem_id"])
print("Dataset:", smoke["dataset"])
print("Action:", smoke["action"])
print("Success:", smoke["success"])
print("Preservation:", round(smoke["preservation"], 3))
print("Forced prefix:", round(smoke["forced_prefix_fraction"], 3))
print("NFE:", smoke["actual_forward_calls"])
print("Generated tokens:", smoke["generated_tokens"])
print("\nREPAIRED OUTPUT\n", smoke["repaired_output"])

assert smoke["generated_tokens"] > 0
assert smoke["actual_forward_calls"] > 0
assert math.isfinite(smoke["wall_seconds"])

print("\nREPAIR SMOKE TEST: PASS")

## 11. Resume-safe full repair sweep

There are 19 semantic failures × 5 actions × 3 seeds = **285 repair trials** by default.

Outputs are flushed to Drive after every trial.

In [ ]:
def trial_id(problem_id, action, seed):
    return f"{problem_id}|{action}|{seed}"

def load_completed_trials(path):
    done = {}
    if not path.exists():
        return done

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            rec = json.loads(line)

            if rec.get("run_signature") != RUN_SIGNATURE:
                raise RuntimeError(
                    "Found trial records from another Notebook 05 configuration."
                )

            done[rec["trial_id"]] = rec

    return done

completed = load_completed_trials(TRIALS_PATH)

TOTAL = (
    len(SELECTED_FAILURES)
    * len(ACTIONS)
    * len(REPAIR_SEEDS)
)

print("Completed:", len(completed), "/", TOTAL)

with open(TRIALS_PATH, "a", encoding="utf-8") as f:
    counter = 0

    for failure in SELECTED_FAILURES:
        for action in ACTIONS:
            for seed in REPAIR_SEEDS:
                counter += 1

                tid = trial_id(
                    failure["problem_id"],
                    action,
                    seed,
                )

                if tid in completed:
                    print(f"[{counter}/{TOTAL}] SKIP {tid}")
                    continue

                try:
                    rec = run_action(
                        failure,
                        action,
                        seed,
                    )
                    rec["trial_id"] = tid
                    rec["status"] = "ok"
                    rec["error"] = None

                except torch.cuda.OutOfMemoryError as e:
                    torch.cuda.empty_cache()
                    gc.collect()

                    rec = {
                        "trial_id": tid,
                        "problem_id": failure["problem_id"],
                        "dataset": failure["dataset"],
                        "action": action,
                        "seed": seed,
                        "status": "oom",
                        "error": str(e),
                        "run_signature": RUN_SIGNATURE,
                    }

                except Exception as e:
                    rec = {
                        "trial_id": tid,
                        "problem_id": failure["problem_id"],
                        "dataset": failure["dataset"],
                        "action": action,
                        "seed": seed,
                        "status": "error",
                        "error": f"{type(e).__name__}: {e}",
                        "run_signature": RUN_SIGNATURE,
                    }

                f.write(
                    json.dumps(rec, ensure_ascii=False)
                    + "\n"
                )
                f.flush()

                if rec["status"] == "ok":
                    print(
                        f"[{counter}/{TOTAL}] "
                        f"{failure['problem_id']} | "
                        f"{action} | seed={seed} | "
                        f"success={rec['success']} | "
                        f"NFE={rec['actual_forward_calls']} | "
                        f"pres={rec['preservation']:.2f}"
                    )
                else:
                    print(
                        f"[{counter}/{TOTAL}] "
                        f"{tid} | {rec['status']} | {rec['error']}"
                    )

print("\nRepair sweep cell finished.")

## 12. Reload, deduplicate, and validate repair trials

In [ ]:
all_trials = []

with open(TRIALS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            all_trials.append(json.loads(line))

current = {
    x["trial_id"]: x
    for x in all_trials
    if x.get("run_signature") == RUN_SIGNATURE
}

expected_ids = {
    trial_id(f["problem_id"], action, seed)
    for f in SELECTED_FAILURES
    for action in ACTIONS
    for seed in REPAIR_SEEDS
}

missing = sorted(expected_ids - set(current))

status_counts = Counter(
    x["status"]
    for tid, x in current.items()
    if tid in expected_ids
)

print("Expected:", len(expected_ids))
print("Found:", len(expected_ids) - len(missing))
print("Statuses:", dict(status_counts))
print("Missing:", len(missing))

if missing:
    print("First missing:", missing[:10])

if status_counts.get("oom", 0):
    raise RuntimeError(
        "OOM records exist. Do not use mixed OOM/success results."
    )

if status_counts.get("error", 0):
    raise RuntimeError(
        "Unhandled repair errors exist. Inspect repair_trials.jsonl."
    )

if missing:
    raise RuntimeError(
        "Repair sweep incomplete. Rerun the sweep cell; completed trials will be skipped."
    )

ok_trials = [
    current[tid]
    for tid in sorted(expected_ids)
]

print("Repair trial integrity: PASS")

## 13. Action-level repair performance

In [ ]:
def wilson(k, n, z=1.959963984540054):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    denom = 1 + z*z/n
    center = (p + z*z/(2*n)) / denom
    half = (
        z
        * math.sqrt(
            p*(1-p)/n + z*z/(4*n*n)
        )
        / denom
    )
    return max(0.0, center-half), min(1.0, center+half)

action_rows = []

for action in ACTIONS:
    rows = [x for x in ok_trials if x["action"] == action]
    k = sum(x["success"] for x in rows)
    n = len(rows)
    lo, hi = wilson(k, n)

    action_rows.append({
        "action": action,
        "trials": n,
        "successes": k,
        "success_rate": k/n,
        "ci95_low": lo,
        "ci95_high": hi,
        "mean_nfe": float(np.mean([
            x["actual_forward_calls"] for x in rows
        ])),
        "median_nfe": float(np.median([
            x["actual_forward_calls"] for x in rows
        ])),
        "mean_preservation": float(np.mean([
            x["preservation"] for x in rows
        ])),
        "mean_wall_seconds": float(np.mean([
            x["wall_seconds"] for x in rows
        ])),
        "max_peak_vram_gb": float(max([
            x["peak_vram_gb"] for x in rows
        ])),
    })

action_df = pd.DataFrame(action_rows).sort_values(
    ["success_rate", "mean_nfe"],
    ascending=[False, True]
)

display(action_df)

## 14. Per-failure repair surface

For each failed example/action pair we estimate empirical success across the 3 repair seeds.

In [ ]:
surface_rows = []

for failure in SELECTED_FAILURES:
    pid = failure["problem_id"]

    for action in ACTIONS:
        rows = [
            x for x in ok_trials
            if x["problem_id"] == pid
            and x["action"] == action
        ]

        assert len(rows) == len(REPAIR_SEEDS)

        success_rate = float(np.mean([
            x["success"] for x in rows
        ]))

        surface_rows.append({
            "problem_id": pid,
            "dataset": failure["dataset"],
            "failure_category":
                failure["verification"]["category"],
            "action": action,
            "success_rate": success_rate,
            "mean_nfe": float(np.mean([
                x["actual_forward_calls"] for x in rows
            ])),
            "mean_preservation": float(np.mean([
                x["preservation"] for x in rows
            ])),
            "mean_wall_seconds": float(np.mean([
                x["wall_seconds"] for x in rows
            ])),
        })

surface_df = pd.DataFrame(surface_rows)

pivot = surface_df.pivot(
    index=["problem_id", "dataset"],
    columns="action",
    values="success_rate",
)

display(pivot)

## 15. Per-instance oracle and Oracle Repair Headroom (ORH)

For each failure, the optimistic oracle chooses:
1. highest empirical repair success rate;
2. then lower mean NFE;
3. then higher preservation.

The **best fixed action** is the one with highest mean repair success across every failure.

`ORH = oracle mean success - best fixed mean success`

This is an oracle diagnostic, not a deployable policy.

In [ ]:
oracle_rows = []

for failure in SELECTED_FAILURES:
    pid = failure["problem_id"]

    cand = surface_df[
        surface_df["problem_id"] == pid
    ].copy()

    cand = cand.sort_values(
        ["success_rate", "mean_nfe", "mean_preservation"],
        ascending=[False, True, False],
    )

    best = cand.iloc[0]

    oracle_rows.append({
        "problem_id": pid,
        "dataset": failure["dataset"],
        "oracle_action": best["action"],
        "oracle_success_rate": best["success_rate"],
        "oracle_mean_nfe": best["mean_nfe"],
        "oracle_preservation": best["mean_preservation"],
    })

oracle_df = pd.DataFrame(oracle_rows)

fixed_df = (
    surface_df.groupby("action", as_index=False)
    .agg(
        mean_success=("success_rate", "mean"),
        mean_nfe=("mean_nfe", "mean"),
        mean_preservation=("mean_preservation", "mean"),
    )
    .sort_values(
        ["mean_success", "mean_nfe"],
        ascending=[False, True],
    )
)

best_fixed = fixed_df.iloc[0]
oracle_mean = float(oracle_df["oracle_success_rate"].mean())
best_fixed_mean = float(best_fixed["mean_success"])
ORH = oracle_mean - best_fixed_mean

action_counts = (
    oracle_df["oracle_action"]
    .value_counts(normalize=True)
    .to_dict()
)

action_concentration = max(action_counts.values())

print("Best fixed action:", best_fixed["action"])
print("Best fixed mean success:", round(best_fixed_mean, 4))
print("Oracle mean success:", round(oracle_mean, 4))
print("Oracle Repair Headroom:", round(ORH, 4))
print("Oracle action distribution:", action_counts)
print("Max action concentration:", round(action_concentration, 4))

display(oracle_df)

## 16. Dataset-specific oracle headroom

In [ ]:
dataset_orh_rows = []

for ds in sorted(surface_df["dataset"].unique()):
    ds_surface = surface_df[
        surface_df["dataset"] == ds
    ]

    ds_oracle = oracle_df[
        oracle_df["dataset"] == ds
    ]

    fixed = (
        ds_surface.groupby("action")["success_rate"]
        .mean()
        .sort_values(ascending=False)
    )

    fixed_action = fixed.index[0]
    fixed_success = float(fixed.iloc[0])
    oracle_success = float(
        ds_oracle["oracle_success_rate"].mean()
    )

    dataset_orh_rows.append({
        "dataset": ds,
        "n_failures": int(
            ds_surface["problem_id"].nunique()
        ),
        "best_fixed_action": fixed_action,
        "best_fixed_success": fixed_success,
        "oracle_success": oracle_success,
        "oracle_headroom": oracle_success - fixed_success,
    })

dataset_orh_df = pd.DataFrame(dataset_orh_rows)
display(dataset_orh_df)

## 17. Success-within-actual-NFE curves

This avoids pretending all actions used the same compute.

A trial counts as successful within budget `B` only when:
- its verifier passes; and
- its **actual recorded forward calls** are ≤ `B`.

This is a conservative cumulative success-vs-compute view.

In [ ]:
NFE_BUDGETS = [24, 48, 72, 96]

budget_rows = []

for action in ACTIONS:
    rows = [
        x for x in ok_trials
        if x["action"] == action
    ]

    for budget in NFE_BUDGETS:
        successes = sum(
            bool(x["success"])
            and x["actual_forward_calls"] <= budget
            for x in rows
        )

        budget_rows.append({
            "action": action,
            "nfe_budget": budget,
            "success_within_budget": successes,
            "trials": len(rows),
            "rate": successes / len(rows),
        })

budget_df = pd.DataFrame(budget_rows)
display(
    budget_df.pivot(
        index="action",
        columns="nfe_budget",
        values="rate",
    )
)

## 18. Pilot Minimal Sufficient Repair Intervention (MSRI)

With only three seeds per action this is **not a certified statistical guarantee**.

For pilot analysis, an action qualifies when:
- empirical success ≥ 2/3;
- mean preservation ≥ 0.50.

Among qualifying actions we select the one with lowest mean NFE.

In [ ]:
PILOT_SUCCESS_THRESHOLD = 2/3
PRESERVATION_THRESHOLD = 0.50

msri_rows = []

for failure in SELECTED_FAILURES:
    pid = failure["problem_id"]

    cand = surface_df[
        (surface_df["problem_id"] == pid)
        & (surface_df["success_rate"] >= PILOT_SUCCESS_THRESHOLD)
        & (surface_df["mean_preservation"] >= PRESERVATION_THRESHOLD)
    ].copy()

    if len(cand) == 0:
        msri_rows.append({
            "problem_id": pid,
            "dataset": failure["dataset"],
            "msri_found": False,
            "action": None,
            "success_rate": None,
            "mean_nfe": None,
            "mean_preservation": None,
        })
        continue

    cand = cand.sort_values(
        ["mean_nfe", "mean_preservation"],
        ascending=[True, False],
    )

    best = cand.iloc[0]

    msri_rows.append({
        "problem_id": pid,
        "dataset": failure["dataset"],
        "msri_found": True,
        "action": best["action"],
        "success_rate": best["success_rate"],
        "mean_nfe": best["mean_nfe"],
        "mean_preservation": best["mean_preservation"],
    })

msri_df = pd.DataFrame(msri_rows)

print(
    "Pilot MSRI coverage:",
    int(msri_df["msri_found"].sum()),
    "/",
    len(msri_df),
)

display(msri_df)

## 19. AR repair heterogeneity gate

This gate asks whether **per-instance repair selection is even potentially useful for the AR baseline**.

It is not the final RepairScope-X diffusion-model GO/NO-GO gate.

Criteria:
- ORH ≥ 0.05;
- no single oracle action selected for >90% of failures;
- at least two datasets show positive oracle headroom;
- no runtime errors or OOMs.

In [ ]:
positive_datasets = int(
    (dataset_orh_df["oracle_headroom"] > 0).sum()
)

AR_GATE = {
    "natural_failures_analyzed": len(SELECTED_FAILURES),
    "repair_trials": len(ok_trials),
    "best_fixed_action": str(best_fixed["action"]),
    "best_fixed_success": best_fixed_mean,
    "oracle_success": oracle_mean,
    "oracle_headroom": ORH,
    "max_oracle_action_concentration": action_concentration,
    "datasets_with_positive_headroom": positive_datasets,
    "oracle_headroom_ge_0_05": bool(ORH >= 0.05),
    "action_concentration_below_0_90": bool(
        action_concentration < 0.90
    ),
    "positive_headroom_on_at_least_two_datasets": bool(
        positive_datasets >= 2
    ),
}

AR_GATE["AR_REPAIR_HETEROGENEITY_PRESENT"] = (
    AR_GATE["oracle_headroom_ge_0_05"]
    and AR_GATE["action_concentration_below_0_90"]
    and AR_GATE[
        "positive_headroom_on_at_least_two_datasets"
    ]
)

print(json.dumps(AR_GATE, indent=2))

## 20. Save tables, gate, and exact analysis artifacts

In [ ]:
action_path = OUT_DIR / "action_summary.csv"
surface_path = OUT_DIR / "repairability_surface.csv"
oracle_path = OUT_DIR / "oracle_actions.csv"
dataset_orh_path = OUT_DIR / "dataset_oracle_headroom.csv"
budget_path = OUT_DIR / "success_within_nfe.csv"
msri_path = OUT_DIR / "pilot_msri.csv"
gate_path = OUT_DIR / "AR_REPAIR_GATE.json"

action_df.to_csv(action_path, index=False)
surface_df.to_csv(surface_path, index=False)
oracle_df.to_csv(oracle_path, index=False)
dataset_orh_df.to_csv(dataset_orh_path, index=False)
budget_df.to_csv(budget_path, index=False)
msri_df.to_csv(msri_path, index=False)

gate_path.write_text(
    json.dumps(AR_GATE, indent=2),
    encoding="utf-8"
)

print("Saved analysis artifacts to:", OUT_DIR)

## 21. Artifact hashes

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)
    return h.hexdigest()

artifact_files = [
    MANIFEST_PATH,
    TRIALS_PATH,
    action_path,
    surface_path,
    oracle_path,
    dataset_orh_path,
    budget_path,
    msri_path,
    gate_path,
]

artifact_manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "run_signature": RUN_SIGNATURE,
    "upstream_run_signature": UPSTREAM_RUN_SIGNATURE,
    "embedded_failure_payload_sha256":
        EMBEDDED_PAYLOAD_SHA256,
    "files": {
        p.name: sha256_file(p)
        for p in artifact_files
        if p.exists()
    },
}

artifact_manifest_path = OUT_DIR / "artifact_manifest.json"
artifact_manifest_path.write_text(
    json.dumps(artifact_manifest, indent=2),
    encoding="utf-8"
)

print(json.dumps(artifact_manifest, indent=2))

## 22. ZIP and download every Notebook 05 result

In [ ]:
import shutil

DOWNLOAD_RESULTS = True

ZIP_BASE = Path(
    "/content/RepairScopeX_Notebook05_AR_Repair_results"
)

zip_path = shutil.make_archive(
    str(ZIP_BASE),
    "zip",
    root_dir=OUT_DIR,
)

print("Created:", zip_path)
print(
    "Size:",
    round(Path(zip_path).stat().st_size / 1024**2, 3),
    "MB",
)

if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(zip_path)

# How to interpret Notebook 05

The key quantity is **Oracle Repair Headroom**.

- If ORH is near zero, a single fixed AR repair action already works about as well as per-instance selection.
- If ORH is materially positive and oracle actions are diverse, the failures have heterogeneous repair structure even for an ordinary AR model.
- This becomes the baseline that the diffusion-model experiments must beat under matched compute and preservation.

The next diffusion-specific experiment should reuse these exact failure definitions, action-cost accounting, verifier semantics, and oracle analysis rather than inventing a different evaluation protocol.